# Week 4 — Pandas Introduction: Sorting, Calculating & value_counts Deep Dive
## Phase 2a Python | PORA Academy Cohort 7

By the end of this session, you will be able to:
- Sort DataFrames with `.sort_values()`
- Add calculated columns
- Use `.nlargest()` / `.nsmallest()`
- Chain pandas operations

---
*Session timing: 2 hours | AI assistance: DeepSeek*

## Why This Session Matters

In the Olist dataset, the order items table contains **112,650 line items** across 99,441 orders — each with a price, a freight charge, and a seller. But raw numbers are not insight. The operations team does not want to scroll through 112,650 rows; they want to know: which items are most expensive, which sellers generate the most revenue, and what proportion of the price is eaten by freight costs.

Today you move from *describing* data to *ranking* and *enriching* it. You will add a `total_cost` column that combines price and freight, classify every item into a price tier, and find the top sellers by total revenue — all with a handful of pandas methods. These are the operations that produce the tables and rankings you see in every business dashboard.

By the time the group exercise begins, you will be able to take a raw CSV and produce a ranked leaderboard of sellers in under ten lines of code. That is the kind of output a manager can act on immediately.

## Setup — Connect to Your Data

Run this cell first. It mounts your Google Drive and loads the order items dataset.

In [ ]:
import pandas as pd
from google.colab import drive
import os

drive.mount('/content/drive')
olist_path = '/content/drive/MyDrive/olist-data'

items = pd.read_csv(os.path.join(olist_path, 'olist_order_items_dataset.csv'))
print("✓ pandas version:", pd.__version__)
print("✓ items shape:", items.shape)
# expected: ✓ items shape: (112650, 7)

## §3 — Concept 1: Loading Items & Adding Calculated Columns

The order items dataset is different from the orders dataset you worked with on Wednesday. Where `orders` had mostly string columns (IDs and statuses), `order_items` is full of numbers — `price`, `freight_value`, `order_item_id`. This means `.describe()` now gives you the meaningful statistics: mean, min, max, and quartiles.

Think of adding a calculated column like filling in a new column in a spreadsheet. In Excel you would click on an empty column header and type a formula in the first cell, then drag it down. In pandas, you assign an expression to a new column name and the operation applies to every row in a single line — no dragging, no copying, works identically whether you have 100 rows or 100,000.

The important difference from a spreadsheet formula is that the pandas column is *permanent* in the DataFrame for the rest of your session. Once you write `items['total_cost'] = items['price'] + items['freight_value']`, every subsequent line of code can reference `items['total_cost']` as if it had always existed in the data.

In [ ]:
# Inspect the items dataset
print(items.shape)
# expected: (112650, 7)

print(items.columns.tolist())
# expected: ['order_id', 'order_item_id', 'product_id', 'seller_id',
#            'shipping_limit_date', 'price', 'freight_value']

# .describe() is meaningful here — price and freight are numeric
print(items[['price', 'freight_value']].describe().round(2))
# expected: price mean=120.65, max=6735.00, min=0.85
#           freight_value mean=19.99, max=409.68

In [ ]:
# Add a calculated column: total cost = price + freight
items['total_cost'] = items['price'] + items['freight_value']

print(items['total_cost'].mean().round(2))
# expected: 140.64

print(f"R${items['total_cost'].sum():,.2f}")
# expected: R$15,843,553.24

# Add a price tier classification using .apply() + lambda
items['price_tier'] = items['price'].apply(
    lambda x: 'Premium' if x >= 500 else 'Standard' if x >= 100 else 'Economy'
)
print(items['price_tier'].value_counts())
# expected:
# Economy     73397
# Standard    30760
# Premium      8493

## §3 — Concept 2: Sorting and Finding Top-N Values

Sorting a DataFrame is as simple as naming the column you want to sort by and choosing whether you want the highest values first (`ascending=False`) or the lowest first (`ascending=True`). Think of it like sorting a leaderboard: you pick the scoring column and decide whether the winner is at the top or the bottom.

`.nlargest(n, column)` and `.nsmallest(n, column)` are a convenience shortcut for "sort and take the top N". They are faster to type than `.sort_values().head()` and they communicate your intent more clearly to someone reading your code — "give me the five most expensive items" reads naturally.

Where these methods become truly powerful is after a `.groupby()`: instead of sorting individual rows, you can aggregate by seller and then rank the sellers themselves. This lets you answer "who are our top 5 revenue generators?" in two lines, collapsing 112,650 rows of transaction data into five seller names and their totals.

In [ ]:
# Sort by price descending — most expensive items first
expensive = items.sort_values('price', ascending=False)
print(expensive[['order_id', 'price', 'freight_value']].head(3))
# expected: top price = 6735.00

In [ ]:
# .nlargest() — top 5 most expensive items
top5_price = items.nlargest(5, 'price')[['product_id', 'price', 'freight_value']]
print(top5_price)
# expected: highest price = 6735.00

# .nsmallest() — 5 cheapest items
bottom5_price = items.nsmallest(5, 'price')[['product_id', 'price']]
print(bottom5_price)
# expected: lowest price = 0.85

In [ ]:
# groupby + nlargest — top 5 sellers by total revenue
seller_revenue = items.groupby('seller_id')['price'].sum()
print(seller_revenue.nlargest(5))
# expected: top seller = R$229,472.63
#           2nd seller = R$222,776.05

---
## 🤖 AI Assistance — DeepSeek Prompts for Today

Use these prompt templates when you get stuck. Remember: attempt the task yourself for 5 minutes first, then validate any generated code against the expected values below.

**Prompt template for calculated columns:**
```
I have a pandas DataFrame called `items` with columns `price` (float) and `freight_value` (float).
How do I add a new column called `total_cost` that equals price + freight_value for each row?
```

**Prompt template for groupby ranking:**
```
I have a pandas DataFrame called `items` with columns `seller_id` (string) and `price` (float).
How do I find the top 5 sellers by total price (sum of all their items)?
Expected: top seller total = 229472.63
```

**Verification rule:** if DeepSeek's answer does not match the expected value, the code is wrong — not the curriculum.

## §4 — Going Deeper: Method Chaining

Once you understand individual pandas operations, you can chain them together — applying one method immediately after another without creating intermediate variables. This is the pandas equivalent of writing a formula-within-a-formula in Excel, but far more readable because each step appears on its own line.

Method chaining is not just about saving lines of code; it makes your *intent* explicit. Reading `items.sort_values('price', ascending=False).head(10)['price'].mean()` left to right tells you exactly what is happening at each step: sort, take top 10, extract price column, take the mean. The alternative — five separate variable assignments — is harder to follow and clutters your notebook with variables you never use again.

In [ ]:
# Method chaining — mean price of the 10 most expensive items
mean_top10_price = (
    items
    .sort_values('price', ascending=False)
    .head(10)
    ['price']
    .mean()
    .round(2)
)
print(mean_top10_price)
# expected: a value well above the overall mean of 120.65

# Chain: filter to Premium tier, then get mean total_cost
premium_avg_cost = (
    items[items['price_tier'] == 'Premium']
    ['total_cost']
    .mean()
    .round(2)
)
print(f"Premium avg total cost: R${premium_avg_cost}")
# expected: significantly above the overall mean of 140.64

## §5 — Common Mistakes

These three mistakes appear in nearly every first attempt at pandas sorting and groupby. Read them now so you recognise the pattern when you hit the error.

In [ ]:
# ── COMMON MISTAKE 1 ────────────────────────────────────────
# WRONG — .groupby() without an aggregation returns a GroupBy object, not numbers:
# result = items.groupby('seller_id')['price']
# print(result)
# → <SeriesGroupBy object> — not useful!

# CORRECT — add an aggregation: .sum(), .mean(), .count(), .max()
result = items.groupby('seller_id')['price'].sum()
print(type(result))
# expected: <class 'pandas.core.series.Series'>
print(result.nlargest(3))
# expected: top 3 seller revenues

# ── COMMON MISTAKE 2 ────────────────────────────────────────
# WRONG — forgetting ascending=False when you want highest first:
# expensive = items.sort_values('price')   ← ascending=True by default!
# print(expensive['price'].head(3))        ← shows cheapest, not most expensive!

# CORRECT:
expensive_correct = items.sort_values('price', ascending=False)
print(expensive_correct['price'].head(3).values)
# expected: values near 6735.00

# ── COMMON MISTAKE 3 ────────────────────────────────────────
# WRONG — modifying the original instead of assigning to a new column:
# items['price'] = items['price'] + items['freight_value']
# ← This overwrites price with total_cost! The original price is gone.

# CORRECT — give the new column a distinct name:
items['total_cost'] = items['price'] + items['freight_value']
print(items[['price', 'freight_value', 'total_cost']].head(2))
# expected: total_cost = price + freight_value

## §6 — Mini-Challenge ⏱ ~8 minutes

Work individually. Use only what you have learned in this session.

**Scenario:** The Olist logistics team wants to identify items where freight is unusually expensive — specifically where `freight_value` is more than **half** the `price` (freight-to-price ratio > 0.5).

**Tasks:**
1. Add a column `freight_ratio` = `freight_value / price`
2. Filter to rows where `freight_ratio > 0.5`
3. Print how many items meet this condition

> There is no single verified expected output — results depend on your exact calculation. A reasonable answer is in the thousands.

In [ ]:
# Mini-Challenge — scaffolded
# Task 1: Add freight_ratio column
# items['freight_ratio'] = ...

# Task 2: Filter to rows where freight_ratio > 0.5
# high_freight = items[...]

# Task 3: Print the count
# print(len(high_freight))


## §7 — Group Exercise (40 minutes)

Work in your groups. Attempt each task yourself first (5 minutes), then use DeepSeek if you are stuck. **Verify every answer against the expected value before moving on.**

Using `olist_order_items_dataset.csv` (already loaded as `items`):

In [ ]:
# ── Task 1 ──────────────────────────────────────────────────
# Total revenue (sum of price column)
# Your code here:
# print(f"R${items['price'].sum():,.2f}")
# expected: R$13,591,643.70

# ── Task 2 ──────────────────────────────────────────────────
# Total freight collected
# Your code here:
# print(f"R${items['freight_value'].sum():,.2f}")
# expected: R$2,251,909.54

# ── Task 3 ──────────────────────────────────────────────────
# Add a column 'revenue_share' = price / total_revenue * 100
# (each item's % contribution to total revenue)
# total_revenue = items['price'].sum()
# items['revenue_share'] = ...
# print(items['revenue_share'].describe().round(4))

# ── Task 4 ──────────────────────────────────────────────────
# How many items are in the "Premium" tier (price >= R$500)?
# Your code here:
# print(len(items[...]))
# expected: 8493

# ── Task 5 ──────────────────────────────────────────────────
# Which order has the most items?
# Hint: groupby order_id, count order_item_id, nlargest(1)
# Your code here:
# expected: max items in one order = 21

# ── Task 6 ──────────────────────────────────────────────────
# What is the average freight as a % of price?
# avg_freight_pct = (items['freight_value'] / items['price']).mean() * 100
# Your code here:
# expected: ~20%


## §8 — Session Summary

| Concept | Key Syntax | Quick Example |
|---|---|---|
| Load numeric CSV | `pd.read_csv()` + `.describe()` | `items.shape` → `(112650, 7)` |
| Add calculated column | `df['new'] = df['a'] + df['b']` | `items['total_cost'] = price + freight` |
| Classify with apply | `df['col'].apply(lambda x: ...)` | Economy / Standard / Premium tiers |
| Sort descending | `.sort_values('col', ascending=False)` | Most expensive item first |
| Top-N rows | `.nlargest(n, 'col')` | Top 5 by price |
| Bottom-N rows | `.nsmallest(n, 'col')` | 5 cheapest items |
| Group and aggregate | `.groupby('col')['val'].sum()` | Revenue per seller |
| Group + rank | `.groupby(...).sum().nlargest(5)` | Top 5 sellers by revenue |
| Chain operations | `df.method1().method2()` | Sort → head → mean |

---

**Weekly Assignment (due before next session):** Submit `week4_assignment.ipynb` covering both the orders and order_items datasets. See the exercises notebook for the full brief.

**Coming up — Week 5**: GroupBy & Aggregation — you will learn to answer "sales by state", "average delivery time by product category", and other multi-dimensional business questions using the full power of `.groupby()` with multiple aggregation functions.